# Step 6: Experiment Tracking with MLflow

**SageMaker Unified Studio Component**: MLflow

**What you'll learn**: Track experiments for reproducibility

In [ ]:
import pandas as pd
import mlflow
import mlflow.sklearn
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from dotenv import load_dotenv

load_dotenv()
bucket_name = os.getenv('BUCKET_NAME')

## Setup MLflow

In [ ]:
mlflow.set_experiment("machine-overheat")
print("MLflow experiment: machine-overheat")

## Load Data

In [ ]:
s3_path = f's3://{bucket_name}/data/features/features.parquet'
df = pd.read_parquet(s3_path)

X = df[['temperature', 'temp_diff']]
y = df['overheat']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## Train and Log with MLflow

In [ ]:
with mlflow.start_run(run_name="logistic_regression_v1"):
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("random_state", 42)
    
    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    
    mlflow.sklearn.log_model(model, "model")
    
    print(f"✓ Run logged with accuracy: {accuracy:.3f}")

## View Experiment Results

The experiments are stored locally in the `mlruns/` folder. Use MLflow's API to view them:

In [ ]:
# Get the experiment
experiment = mlflow.get_experiment_by_name("machine-overheat")
print(f"Experiment: {experiment.name}")
print(f"Experiment ID: {experiment.experiment_id}")
print(f"Artifact Location: {experiment.artifact_location}")
print()

# Get all runs for this experiment
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
print("=== Logged Runs ===")
display(runs[['run_id', 'status', 'start_time', 'metrics.accuracy', 'metrics.precision', 'metrics.recall', 'metrics.f1_score', 'params.model_type']])